# LAB02 Practica 3

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

# Datos de entrada

In [29]:
# Cargar los datos
df = pd.read_csv("got_train.csv")

# Seleccionar características y objetivo
features = ["male", "book1", "book2", "book3", "book4", "book5", "isMarried", "isNoble", "numDeadRelations", "isPopular"]
target = "alive"

X = df[features].values
y = df[target].values

# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalizar los datos
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convertir datos a tensores de PyTorch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Crear DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)



# Definimos estructura del MLP

In [30]:
#Definimos los hiperparámetros
input_size = X_train.shape[1]
learning_rate = 0.01

# Definir la estructura de la red MLP
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        # Capas de la red
        self.capa1 = nn.Linear(input_size, 32)
        self.capa2 = nn.Linear(32, 64)
        self.capa3 = nn.Linear(64, 32)
        self.capa4 = nn.Linear(32, 1)

        # Función de activación
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        # Establecemos un dropout para evitar que no hayan neuronas que acaben siendo insignificantes
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        x = self.relu(self.capa1(x))
        x = self.dropout(x)
        x = self.relu(self.capa2(x))
        x = self.dropout(x)
        x = self.relu(self.capa3(x))
        x = self.dropout(x)
        x = self.capa4(x)
        x = self.sigmoid(x)  # Aplicamos Sigmoid para obtener un valor entre 0 y 1
        return x

# Inicializar el modelo
input_dim = X_train.shape[1]
model = MLP(input_dim)

# Definir función de pérdida y optimizador
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Entrenamiento

In [31]:
# Entrenamiento del modelo
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_loader):.4f}")



Epoch [1/100], Loss: 0.0816
Epoch [2/100], Loss: 0.0712
Epoch [3/100], Loss: 0.0704
Epoch [4/100], Loss: 0.0700
Epoch [5/100], Loss: 0.0652
Epoch [6/100], Loss: 0.0646
Epoch [7/100], Loss: 0.0642
Epoch [8/100], Loss: 0.0661
Epoch [9/100], Loss: 0.0647
Epoch [10/100], Loss: 0.0651
Epoch [11/100], Loss: 0.0666
Epoch [12/100], Loss: 0.0629
Epoch [13/100], Loss: 0.0640
Epoch [14/100], Loss: 0.0628
Epoch [15/100], Loss: 0.0638
Epoch [16/100], Loss: 0.0648
Epoch [17/100], Loss: 0.0618
Epoch [18/100], Loss: 0.0658
Epoch [19/100], Loss: 0.0627
Epoch [20/100], Loss: 0.0629
Epoch [21/100], Loss: 0.0631
Epoch [22/100], Loss: 0.0639
Epoch [23/100], Loss: 0.0664
Epoch [24/100], Loss: 0.0644
Epoch [25/100], Loss: 0.0638
Epoch [26/100], Loss: 0.0647
Epoch [27/100], Loss: 0.0649
Epoch [28/100], Loss: 0.0657
Epoch [29/100], Loss: 0.0660
Epoch [30/100], Loss: 0.0647
Epoch [31/100], Loss: 0.0628
Epoch [32/100], Loss: 0.0642
Epoch [33/100], Loss: 0.0639
Epoch [34/100], Loss: 0.0652
Epoch [35/100], Loss: 0

# Evaluación

In [32]:
# Predicciones continuas del modelo
y_pred_test = model(X_test_tensor).detach().numpy()

# Calcular MSE
mse = mean_squared_error(y_test, y_pred_test)
print(f"Test MSE: {mse:.4f}")

Test MSE: 0.0616


# Predicción

In [33]:
# 1. Cargar los datos de predicción
df_predict = pd.read_csv('got_predict.csv')

# Seleccionar las mismas características que en el entrenamiento
features = ["male", "book1", "book2", "book3", "book4", "book5", "isMarried", "isNoble", "numDeadRelations", "popularity"]

X_predict = df_predict[features].values

# 2. Normalizar los datos (usando el scaler ya entrenado)
X_predict = scaler.transform(X_predict)

# Convertir los datos de predicción a tensor
X_predict_tensor = torch.tensor(X_predict, dtype=torch.float32)

# 3. Realizar las predicciones con el modelo entrenado
model.eval()  # Asegurarse de que el modelo está en modo de evaluación

# Obtener las probabilidades de vivir
with torch.no_grad():
    y_pred_predict = model(X_predict_tensor).numpy()

# Calcular la probabilidad de morir (1 - probabilidad de vivir)
#probabilidad_de_morir = 1 - y_pred_predict       



# 4. Añadir las probabilidades de morir al dataframe
df_predict['predicted_mortality'] = y_pred_predict

# 5. Mostrar la probabilidad de morir para todos los personajes
print("Probabilidades de morir para todos los personajes:")
print(df_predict[['name', 'predicted_mortality']])

# 6. Obtener el personaje con la mayor probabilidad de morir
most_likely_to_die = df_predict.loc[df_predict['predicted_mortality'].idxmax()]

# Mostrar el personaje con la mayor probabilidad de morir
print(f"\nEl personaje con más probabilidades de morir es {most_likely_to_die['name']} con una probabilidad de {most_likely_to_die['predicted_mortality']:.4f}.")


Probabilidades de morir para todos los personajes:
                            name  predicted_mortality
0               Tommen Baratheon             0.013020
1             Daenerys Targaryen             0.113136
2                      Coldhands             0.677183
3                 Othell Yarwyck             0.638184
4  Roland Crakehall (Kingsguard)             0.593589

El personaje con más probabilidades de morir es Coldhands con una probabilidad de 0.6772.
